# W3D4_LAB_Slice_docs_the_way_Google_wishes_you_would
- Week 3 / Day 4
- Student: Andreas Papachristophorou
- Course: AI Consulting & Integration 2026-07
- Date: 2026-07-23

## Step 1: Setup and Data Loading

In [ ]:
import os
import json
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydub import AudioSegment
from tempfile import TemporaryDirectory

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set")

client = OpenAI(api_key=api_key)

# 1. Point and load the audio file
audio_dir = Path("sources")
audio_filename = "The_Blueprint_For_Trustworthy_AI.m4a"
audio_path = audio_dir / audio_filename

if audio_path.exists():
    print("The audio file is ready.")
    print("Audio path:", audio_path)
else:
    print("The audio file was not found. Put it inside the 'sources' folder.")

# 2. Function that sends the audio file with extra context (prompt)
def transcribe_with_prompt(audio_path: Path, prompt_text: str, max_chunk_mb: int = 24) -> str:
    """Transcribe audio, splitting it into chunks if the file is too large."""
    audio = AudioSegment.from_file(audio_path)
    max_chunk_bytes = max_chunk_mb * 1024 * 1024
    audio_file_size = audio_path.stat().st_size

    def transcribe_file(file_path: Path) -> str:
        with open(file_path, "rb") as file:
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=file,
                prompt=prompt_text,
            )
        return transcript.text.strip()

    # Whisper uploads are limited to 25 MiB, so keep a small safety margin.
    if audio_file_size <= max_chunk_bytes:
        return transcribe_file(audio_path)

    # Chunk by duration using a conservative estimate based on the source file.
    chunk_duration_ms = max(60_000, int(len(audio) * (max_chunk_bytes / audio_file_size) * 0.9))
    transcripts = []

    with TemporaryDirectory() as tmp_dir:
        tmp_dir = Path(tmp_dir)
        for index, start_ms in enumerate(range(0, len(audio), chunk_duration_ms), start=1):
            chunk = audio[start_ms : start_ms + chunk_duration_ms]
            chunk_path = tmp_dir / f"chunk_{index:03d}.m4a"
            chunk.export(chunk_path, format="ipod")

            if chunk_path.stat().st_size > max_chunk_bytes:
                raise ValueError(
                    f"Chunk {index} is still too large to upload: {chunk_path.stat().st_size} bytes"
                )

            chunk_text = transcribe_file(chunk_path)
            if chunk_text:
                transcripts.append(chunk_text)

    return "\n".join(transcripts)

# 3. Add any useful words, names, or technical terms
prompt_text = (
    "Talk or interview about trustworthy AI and responsible AI systems. "
    "Topics include AI risk, AI safety, governance, regulations, ethics, "
    "transparency, accountability, fairness, bias, and trust in AI. "
    "May mention large language models, foundation models, compliance, "
    "data protection, and alignment."
)

# 4. Run the transcription with the prompt
prompted_text = transcribe_with_prompt(audio_path, prompt_text)

print("Prompted transcription:")
print("-" * 50)
print(prompted_text)

# 5. Save transcript to /sources as txt
output_filename = "The_Blueprint_For_Trustworthy_AI_transcript.txt"
output_path = audio_dir / output_filename

with open(output_path, "w", encoding="utf-8") as f:
    f.write(prompted_text)

print("Transcription saved to:", output_path)



The audio file is ready.
Audio path: sources\The_Blueprint_For_Trustworthy_AI.m4a
Prompted transcription:
--------------------------------------------------
So imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. You just drive. You trust the bridge. You trust the engineering standards, the inspections, the laws that say this thing won't fail. Right. It's trust in the infrastructure. It's invisible, but it's there. Exactly. But now let's switch gears. Think about the algorithm that just denied your mortgage application or the AI system scanning your face at the airport. Do you have that same trust? And I mean, honestly, should you? That is the defining question of our decade, I think. We've moved past the wow phase of AI and straight into the wait a minute phase. Today we are digging into the bluep

In [3]:
#stabforming and savign the .pdf file into .txt
from pathlib import Path
from PyPDF2 import PdfReader

sources_dir = Path("sources")
pdf_path = sources_dir / "385082eng.pdf"
txt_path = sources_dir / "385082eng_text.txt"

# Read PDF
reader = PdfReader(pdf_path)
pages_text = []

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        pages_text.append(page_text)

full_text = "\n\n".join(pages_text)

# Save to txt
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(full_text)

print("Saved PDF text to:", txt_path)
print("Number of characters:", len(full_text))

Saved PDF text to: sources\385082eng_text.txt
Number of characters: 22101


## Step 2: Implement Fixed-Size Chunking

In [4]:
sources_dir = Path("sources")

podcast_txt_path = sources_dir / "The_Blueprint_For_Trustworthy_AI_transcript.txt"
pdf_txt_path = sources_dir / "385082eng_text.txt" 

with open(podcast_txt_path, "r", encoding="utf-8") as f:
    podcast_text = f.read()

with open(pdf_txt_path, "r", encoding="utf-8") as f:
    pdf_text = f.read()

print("Podcast characters:", len(podcast_text))
print("PDF characters:", len(pdf_text))

Podcast characters: 16204
PDF characters: 22101


In [5]:
# Use CharacterTextSplitter with fixed chunk size
# test run at 500

from langchain_text_splitters import CharacterTextSplitter

fixed_splitter_500 = CharacterTextSplitter(
    separator="",      # try to split on paragraph breaks when possible
    chunk_size=500,        # target ~500 characters per chunk
    chunk_overlap=0,       # no overlap
    length_function=len,   # measure length by characters
    is_separator_regex=False
)

fixed_chunks_500_podcast = fixed_splitter_500.split_text(podcast_text)
fixed_chunks_500_pdf = fixed_splitter_500.split_text(pdf_text)

print("Podcast chunks (fixed 500):", len(fixed_chunks_500_podcast))
print("PDF chunks (fixed 500):", len(fixed_chunks_500_pdf))


Podcast chunks (fixed 500): 33
PDF chunks (fixed 500): 45


In [6]:
# Use CharacterTextSplitter with fixed chunk size
# Loop run with different sizes: 500, 1000, 2000 char. and overlap values: 0, 50, 100 char.

from langchain_text_splitters import CharacterTextSplitter

configs = [
    (500, 0),
    (500, 50),
    (500, 100),
    (1000, 0),
    (1000, 50),
    (2000, 0),
]

fixed_results = []

for chunk_size, chunk_overlap in configs:
    splitter = CharacterTextSplitter(
        separator="",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )

    podcast_chunks = splitter.split_text(podcast_text)
    pdf_chunks = splitter.split_text(pdf_text)

    avg_podcast_len = sum(len(c) for c in podcast_chunks) / len(podcast_chunks)
    avg_pdf_len = sum(len(c) for c in pdf_chunks) / len(pdf_chunks)

    fixed_results.append({
        "Chunk size": chunk_size,
        "Overlap": chunk_overlap,
        "# Podcast chunks": len(podcast_chunks),
        "Avg podcast len (chars)": avg_podcast_len,
        "# PDF chunks": len(pdf_chunks),
        "Avg PDF len (chars)": avg_pdf_len,
    })

fixed_results

[{'Chunk size': 500,
  'Overlap': 0,
  '# Podcast chunks': 33,
  'Avg podcast len (chars)': 490.6363636363636,
  '# PDF chunks': 45,
  'Avg PDF len (chars)': 490.8666666666667},
 {'Chunk size': 500,
  'Overlap': 50,
  '# Podcast chunks': 36,
  'Avg podcast len (chars)': 498.30555555555554,
  '# PDF chunks': 50,
  'Avg PDF len (chars)': 490.58},
 {'Chunk size': 500,
  'Overlap': 100,
  '# Podcast chunks': 41,
  'Avg podcast len (chars)': 492.390243902439,
  '# PDF chunks': 56,
  'Avg PDF len (chars)': 492.57142857142856},
 {'Chunk size': 1000,
  'Overlap': 0,
  '# Podcast chunks': 17,
  'Avg podcast len (chars)': 952.7647058823529,
  '# PDF chunks': 23,
  'Avg PDF len (chars)': 960.6521739130435},
 {'Chunk size': 1000,
  'Overlap': 50,
  '# Podcast chunks': 18,
  'Avg podcast len (chars)': 947.1666666666666,
  '# PDF chunks': 24,
  'Avg PDF len (chars)': 968.2916666666666},
 {'Chunk size': 2000,
  'Overlap': 0,
  '# Podcast chunks': 9,
  'Avg podcast len (chars)': 1800.0,
  '# PDF chunk

In [7]:
# Display as a nicely formatted table with pandas
import pandas as pd

fixed_df = pd.DataFrame(fixed_results)

# Optional: round averages for display
fixed_df["Avg podcast len (chars)"] = fixed_df["Avg podcast len (chars)"].round(1)
fixed_df["Avg PDF len (chars)"] = fixed_df["Avg PDF len (chars)"].round(1)

fixed_df

,Chunk size,Overlap,# Podcast chunks,Avg podcast len (chars),# PDF chunks,Avg PDF len (chars)
0,500,0,33,490.6,45,490.9
1,500,50,36,498.3,50,490.6
2,500,100,41,492.4,56,492.6
3,1000,0,17,952.8,23,960.7
4,1000,50,18,947.2,24,968.3
5,2000,0,9,1800.0,12,1841.5


In [8]:
# Savign the results in a markdown format at reports/

import pandas as pd  # if not already imported
from pathlib import Path

# 1. Generate Markdown from the DataFrame
def fixed_df_to_markdown(df: pd.DataFrame) -> str:
    header = (
        "| Chunk size | Overlap | # Podcast chunks | Avg podcast len (chars) "
        "| # PDF chunks | Avg PDF len (chars) |\n"
    )
    header += (
        "|-----------|---------|------------------|-------------------------"
        "|--------------|----------------------|\n"
    )

    rows = []
    for _, r in df.iterrows():
        row = (
            f"| {int(r['Chunk size'])} "
            f"| {int(r['Overlap'])} "
            f"| {int(r['# Podcast chunks'])} "
            f"| {r['Avg podcast len (chars)']:.1f} "
            f"| {int(r['# PDF chunks'])} "
            f"| {r['Avg PDF len (chars)']:.1f} |"
        )
        rows.append(row)

    return header + "\n".join(rows)

fixed_table_md = fixed_df_to_markdown(fixed_df)
print(f"Preview first ~100 chars:\n{fixed_table_md[:100]}")


# 2. Define the reports directory
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)  # create if it doesn't exist

# 3. Choose a descriptive filename
md_path = reports_dir / "fixed_size_chunking_podcast_vs_pdf.md"

# 4. Write the markdown content
with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Fixed-Size Chunking Results (Podcast vs PDF)\n\n")
    f.write("This table summarizes the number and average length of chunks for different "
            "fixed-size and overlap settings, comparing podcast transcript and PDF text.\n\n")
    f.write(fixed_table_md)
    f.write("\n")

print(f"\nSaved Markdown table to:", md_path)

Preview first ~100 chars:
| Chunk size | Overlap | # Podcast chunks | Avg podcast len (chars) | # PDF chunks | Avg PDF len (ch

Saved Markdown table to: reports\fixed_size_chunking_podcast_vs_pdf.md


Analysis 

In [9]:
def show_chunk_endings(name, chunks, n=5):
    print(f"\n=== {name} (showing last 100 chars of first {n} chunks) ===")
    for i, ch in enumerate(chunks[:n], start=1):
        print(f"\n--- Chunk {i} ---")
        print(ch[-100:])  # last 100 characters

# Example: inspect fixed 500-char chunks we already created
show_chunk_endings("Podcast fixed 500", fixed_chunks_500_podcast, n=5)
show_chunk_endings("PDF fixed 500", fixed_chunks_500_pdf, n=5)



=== Podcast fixed 500 (showing last 100 chars of first 5 chunks) ===

--- Chunk 1 ---
 the infrastructure. It's invisible, but it's there. Exactly. But now let's switch gears. Think abou

--- Chunk 2 ---
rguably the Magna Carta for ethical computing, the ethics guidelines for trustworthy AI. This is a h

--- Chunk 3 ---
plosion we have today. So why are we blowing the dust off this specific report? Because it's not dus

--- Chunk 4 ---
act concept like fairness and turn it into, I don't know, Python code. We've got three pillars, four

--- Chunk 5 ---
of it like a three-legged stool. The first leg is lawful. The AI has to comply with all the regulati

=== PDF fixed 500 (showing last 100 chars of first 5 chunks) ===

--- Chunk 1 ---
al-ShareAlike 3.0 IGO (CC-BY-NC-SA 3.0 IGO) license 
(http://creativecommons.org/licenses/by-nc-sa/3

--- Chunk 2 ---
nciples lay out a Human Rights-centred 
Approach to the Ethics of AI·Message from Gabriela Ramos
Ass

--- Chunk 3 ---
 unique mandate, UNES

In [10]:
# Function to capture chunk endings as a Markdown-like string
def chunk_endings_markdown(name: str, chunks, n: int = 5, tail_chars: int = 100) -> str:
    lines = []
    lines.append(f"## {name} (last {tail_chars} characters of first {n} chunks)\n")
    for i, ch in enumerate(chunks[:n], start=1):
        lines.append(f"### Chunk {i}\n")
        # Use fenced code block for readability in Markdown
        lines.append("```text")
        lines.append(ch[-tail_chars:])
        lines.append("```")
        lines.append("")  # blank line
    return "\n".join(lines)

podcast_endings_md = chunk_endings_markdown("Podcast fixed 500", fixed_chunks_500_podcast, n=5)
pdf_endings_md = chunk_endings_markdown("PDF fixed 500", fixed_chunks_500_pdf, n=5)

# Save both inspections into a single .md report

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)

md_path = reports_dir / "fixed_chunking_boundary_examples.md"

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Fixed-Size Chunking Boundary Examples\n\n")
    f.write("This file shows the last 100 characters of the first few chunks "
            "for the podcast transcript and the PDF text using the fixed-size "
            "500-character, 0-overlap configuration.\n\n")
    f.write(podcast_endings_md)
    f.write("\n\n")
    f.write(pdf_endings_md)
    f.write("\n")

print("Saved chunk boundary examples to:", md_path)


Saved chunk boundary examples to: reports\fixed_chunking_boundary_examples.md


## Step 3: Implement Recursive Character Chunking

In [11]:
# Basic recursive splitter setup
from langchain_text_splitters import RecursiveCharacterTextSplitter

# test
recursive_splitter_500 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    # Let it use the default separator priority:
    # ["\n\n", "\n", " ", ""]
)

recursive_chunks_500_podcast = recursive_splitter_500.split_text(podcast_text)
recursive_chunks_500_pdf = recursive_splitter_500.split_text(pdf_text)

len(recursive_chunks_500_podcast), len(recursive_chunks_500_pdf)


(33, 54)

In [12]:
# Experiment with multiple chunk sizes (500, 1000, 2000)
recursive_configs = [
    (500, 0),
    (1000, 0),
    (2000, 0),
]

recursive_results = []

for chunk_size, chunk_overlap in recursive_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    podcast_chunks = splitter.split_text(podcast_text)
    pdf_chunks = splitter.split_text(pdf_text)

    avg_podcast_len = sum(len(c) for c in podcast_chunks) / len(podcast_chunks)
    avg_pdf_len = sum(len(c) for c in pdf_chunks) / len(pdf_chunks)

    recursive_results.append({
        "Chunk size": chunk_size,
        "Overlap": chunk_overlap,
        "# Podcast chunks": len(podcast_chunks),
        "Avg podcast len (chars)": avg_podcast_len,
        "# PDF chunks": len(pdf_chunks),
        "Avg PDF len (chars)": avg_pdf_len,
    })

In [13]:
recursive_df = pd.DataFrame(recursive_results)
recursive_df["Avg podcast len (chars)"] = recursive_df["Avg podcast len (chars)"].round(1)
recursive_df["Avg PDF len (chars)"] = recursive_df["Avg PDF len (chars)"].round(1)
recursive_df

,Chunk size,Overlap,# Podcast chunks,Avg podcast len (chars),# PDF chunks,Avg PDF len (chars)
0,500,0,33,490.1,54,407.3
1,1000,0,17,952.2,30,734.7
2,2000,0,9,1799.6,17,1297.9


In [14]:
import pandas as pd

def recursive_df_to_markdown(df: pd.DataFrame) -> str:
    header = (
        "| Chunk size (chars) | Overlap (chars) | # Podcast chunks | Avg podcast len (chars) "
        "| # PDF chunks | Avg PDF len (chars) |\n"
    )
    header += (
        "|--------------------|-----------------|------------------|-------------------------"
        "|--------------|----------------------|\n"
    )

    rows = []
    for _, r in df.iterrows():
        row = (
            f"| {int(r['Chunk size'])} "
            f"| {int(r['Overlap'])} "
            f"| {int(r['# Podcast chunks'])} "
            f"| {r['Avg podcast len (chars)']:.1f} "
            f"| {int(r['# PDF chunks'])} "
            f"| {r['Avg PDF len (chars)']:.1f} |"
        )
        rows.append(row)

    return header + "\n".join(rows)

recursive_table_md = recursive_df_to_markdown(recursive_df)
print(recursive_table_md[:500])  # quick preview

| Chunk size (chars) | Overlap (chars) | # Podcast chunks | Avg podcast len (chars) | # PDF chunks | Avg PDF len (chars) |
|--------------------|-----------------|------------------|-------------------------|--------------|----------------------|
| 500 | 0 | 33 | 490.1 | 54 | 407.3 |
| 1000 | 0 | 17 | 952.2 | 30 | 734.7 |
| 2000 | 0 | 9 | 1799.6 | 17 | 1297.9 |


In [15]:
# Adjust separator priority (podcast vs PDF)

# Podcast: prioritize sentences and words
podcast_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    separators=["\n\n", "\n", ". ", " ", ""]
)

podcast_chunks_custom = podcast_splitter.split_text(podcast_text)
print("Podcast chunks (custom separators):", len(podcast_chunks_custom))
print("Podcast sample chunk end:\n", podcast_chunks_custom[0][-50:])

# PDF: prioritize big section and paragraph breaks
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    separators=["\n\n\n", "\n\n", "\n", ". ", " ", ""]
)

pdf_chunks_custom = pdf_splitter.split_text(pdf_text)
print("\nPDF chunks (custom separators):", len(pdf_chunks_custom))
print("PDF sample chunk end:\n", pdf_chunks_custom[0][-50:])

Podcast chunks (custom separators): 36
Podcast sample chunk end:
 ut it's there. Exactly. But now let's switch gears

PDF chunks (custom separators): 54
PDF sample chunk end:
 e Ethics
 of Artificial
Intelligence
Supported by:


In [16]:
# Compute stats for 500-char custom separators

avg_podcast_len_custom = sum(len(c) for c in podcast_chunks_custom) / len(podcast_chunks_custom)
avg_pdf_len_custom = sum(len(c) for c in pdf_chunks_custom) / len(pdf_chunks_custom)

print("Podcast 500/custom - #chunks:", len(podcast_chunks_custom),
      "| Avg len (chars):", round(avg_podcast_len_custom, 1))
print("PDF 500/custom      - #chunks:", len(pdf_chunks_custom),
      "| Avg len (chars):", round(avg_pdf_len_custom, 1))

Podcast 500/custom - #chunks: 36 | Avg len (chars): 450.1
PDF 500/custom      - #chunks: 54 | Avg len (chars): 407.3


In [17]:
def recursive_custom_summary_table_md(
    podcast_chunks, pdf_chunks, avg_podcast_len, avg_pdf_len
) -> str:
    header = (
        "| Content type | Chunk size (chars) | Overlap (chars) | Separator priority "
        "| # Chunks | Avg len (chars) |\n"
    )
    header += (
        "|-------------|--------------------|-----------------|--------------------"
        "|-----------|-----------------|\n"
    )

    podcast_row = (
        "| Podcast | 500 | 0 | \\n\\n > \\n > '. ' > ' ' > '' "
        f"| {len(podcast_chunks)} | {avg_podcast_len:.1f} |\n"
    )

    pdf_row = (
        "| PDF | 500 | 0 | \\n\\n\\n > \\n\\n > \\n > '. ' > ' ' > '' "
        f"| {len(pdf_chunks)} | {avg_pdf_len:.1f} |\n"
    )

    return header + podcast_row + pdf_row

recursive_custom_table_md = recursive_custom_summary_table_md(
    podcast_chunks_custom,
    pdf_chunks_custom,
    avg_podcast_len_custom,
    avg_pdf_len_custom,
)

print(recursive_custom_table_md)

| Content type | Chunk size (chars) | Overlap (chars) | Separator priority | # Chunks | Avg len (chars) |
|-------------|--------------------|-----------------|--------------------|-----------|-----------------|
| Podcast | 500 | 0 | \n\n > \n > '. ' > ' ' > '' | 36 | 450.1 |
| PDF | 500 | 0 | \n\n\n > \n\n > \n > '. ' > ' ' > '' | 54 | 407.3 |



In [18]:
# Reuse the shared chunk_endings_markdown helper

podcast_recursive_custom_endings_md = chunk_endings_markdown(
    "Podcast recursive custom 500", podcast_chunks_custom, n=5
)

pdf_recursive_custom_endings_md = chunk_endings_markdown(
    "PDF recursive custom 500", pdf_chunks_custom, n=5
)
# quick test 
print(podcast_recursive_custom_endings_md[:100])
print(pdf_recursive_custom_endings_md[:100])

# saving into a markdown file
reports_dir = Path("reports") 
reports_dir.mkdir(exist_ok=True) 

md_path = reports_dir / "recursive_chunking_boundary_examples_custom.md"

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Recursive Chunking Boundary Examples (Custom Separators)\n\n")
    f.write("Last 100 chars of the first few recursive chunks for podcast and PDF "
            "with custom separator priorities.\n\n")
    f.write(podcast_recursive_custom_endings_md)
    f.write("\n\n")
    f.write(pdf_recursive_custom_endings_md)
    f.write("\n")

print("Saved recursive boundary examples (custom) to:", md_path)

## Podcast recursive custom 500 (last 100 characters of first 5 chunks)

### Chunk 1

```text
t's tr
## PDF recursive custom 500 (last 100 characters of first 5 chunks)

### Chunk 1

```text
 2021
Key 
Saved recursive boundary examples (custom) to: reports\recursive_chunking_boundary_examples_custom.md


## Step 4: Implement Token-Based Chunking

In [19]:
# Token 500-char split
from langchain_text_splitters import TokenTextSplitter

token_splitter_500 = TokenTextSplitter(
    encoding_name="cl100k_base",  # GPT-4 / GPT-4o style tokenizer
    chunk_size=500,
    chunk_overlap=0,
)

token_chunks_500_podcast = token_splitter_500.split_text(podcast_text)
token_chunks_500_pdf = token_splitter_500.split_text(pdf_text)

print("Podcast token chunks (500):", len(token_chunks_500_podcast))
print("PDF token chunks (500):", len(token_chunks_500_pdf))


Podcast token chunks (500): 7
PDF token chunks (500): 10


In [20]:
# Experiment with multiple token sizes and overlaps

token_configs = [
    (500, 0),
    (1000, 0),
    (2000, 0),
]

token_results = []

for chunk_size, chunk_overlap in token_configs:
    splitter = TokenTextSplitter(
        encoding_name="cl100k_base",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    podcast_chunks = splitter.split_text(podcast_text)
    pdf_chunks = splitter.split_text(pdf_text)

    avg_podcast_len = sum(len(c) for c in podcast_chunks) / len(podcast_chunks)
    avg_pdf_len = sum(len(c) for c in pdf_chunks) / len(pdf_chunks)

    token_results.append({
        "Chunk size (tokens)": chunk_size,
        "Overlap (tokens)": chunk_overlap,
        "# Podcast chunks": len(podcast_chunks),
        "Avg podcast len (chars)": avg_podcast_len,
        "# PDF chunks": len(pdf_chunks),
        "Avg PDF len (chars)": avg_pdf_len,
    })

In [21]:
# Display token-based results as a DataFrame

token_df = pd.DataFrame(token_results)
token_df["Avg podcast len (chars)"] = token_df["Avg podcast len (chars)"].round(1)
token_df["Avg PDF len (chars)"] = token_df["Avg PDF len (chars)"].round(1)

token_df

,Chunk size (tokens),Overlap (tokens),# Podcast chunks,Avg podcast len (chars),# PDF chunks,Avg PDF len (chars)
0,500,0,7,2314.9,10,2210.1
1,1000,0,4,4051.0,5,4420.2
2,2000,0,2,8102.0,3,7367.0


In [22]:
# Inspect token 500 boundaries

token_podcast_endings_md = chunk_endings_markdown(
    "Podcast token 500",
    token_chunks_500_podcast,
    n=5
)

token_pdf_endings_md = chunk_endings_markdown(
    "PDF token 500",
    token_chunks_500_pdf,
    n=5
)


In [23]:
# Convert token_df to a Markdown table string

def token_df_to_markdown(df: pd.DataFrame) -> str:
    header = (
        "| Chunk size (tokens) | Overlap (tokens) | # Podcast chunks | Avg podcast len (chars) "
        "| # PDF chunks | Avg PDF len (chars) |\n"
    )
    header += (
        "|---------------------|------------------|------------------|-------------------------"
        "|--------------|----------------------|\n"
    )

    rows = []
    for _, r in df.iterrows():
        row = (
            f"| {int(r['Chunk size (tokens)'])} "
            f"| {int(r['Overlap (tokens)'])} "
            f"| {int(r['# Podcast chunks'])} "
            f"| {r['Avg podcast len (chars)']:.1f} "
            f"| {int(r['# PDF chunks'])} "
            f"| {r['Avg PDF len (chars)']:.1f} |"
        )
        rows.append(row)

    return header + "\n".join(rows)

token_table_md = token_df_to_markdown(token_df)
print(token_table_md[:500])  # quick preview

| Chunk size (tokens) | Overlap (tokens) | # Podcast chunks | Avg podcast len (chars) | # PDF chunks | Avg PDF len (chars) |
|---------------------|------------------|------------------|-------------------------|--------------|----------------------|
| 500 | 0 | 7 | 2314.9 | 10 | 2210.1 |
| 1000 | 0 | 4 | 4051.0 | 5 | 4420.2 |
| 2000 | 0 | 2 | 8102.0 | 3 | 7367.0 |


In [24]:
# Save table + boundaries into one .md report

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)

md_path = reports_dir / "token_chunking_results_and_boundaries.md"

with open(md_path, "w", encoding="utf-8") as f:
    # Title and intro
    f.write("# Token-Based Chunking Results (Podcast vs PDF)\n\n")
    f.write(
        "This report summarizes token-based chunking statistics and shows example chunk boundaries "
        "for the podcast transcript and the PDF text.\n\n"
    )

    # Table section
    f.write("## Summary table\n\n")
    f.write(token_table_md)
    f.write("\n\n")

    # Boundary examples section
    f.write("## Boundary examples\n\n")
    f.write(token_podcast_endings_md)
    f.write("\n\n")
    f.write(token_pdf_endings_md)
    f.write("\n")

print("Saved token-based chunking report to:", md_path)

Saved token-based chunking report to: reports\token_chunking_results_and_boundaries.md


## Step 4: Implement Token-Based Chunking

In [25]:
# Recursive 1000-char split for PDF
recursive_splitter_1000_pdf = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,
)

recursive_chunks_1000_pdf = recursive_splitter_1000_pdf.split_text(pdf_text)

print("Recursive 1000 PDF chunks:", len(recursive_chunks_1000_pdf))

Recursive 1000 PDF chunks: 30


In [26]:
def chunk_stats(name, chunks):
    lengths = [len(c) for c in chunks]
    return {
        "Strategy": name,
        "Avg chunk size (chars)": sum(lengths) / len(lengths),
        "Min chunk size (chars)": min(lengths),
        "Max chunk size (chars)": max(lengths),
        "# of chunks": len(lengths),
    }

# Use the shared 500-char and 500-token splitters
fixed_chunks_500_pdf = fixed_splitter_500.split_text(pdf_text)
token_chunks_500_pdf = token_splitter_500.split_text(pdf_text)

# Now create the stats list
stats_list_pdf = [
    chunk_stats("PDF fixed 500", fixed_chunks_500_pdf),
    chunk_stats("PDF recursive 1000", recursive_chunks_1000_pdf),
    chunk_stats("PDF token 500", token_chunks_500_pdf),
]

stats_df_pdf = pd.DataFrame(stats_list_pdf)
stats_df_pdf["Avg chunk size (chars)"] = stats_df_pdf["Avg chunk size (chars)"].round(1)

stats_df_pdf


,Strategy,Avg chunk size (chars),Min chunk size (chars),Max chunk size (chars),# of chunks
0,PDF fixed 500,490.9,101,500,45
1,PDF recursive 1000,734.7,189,992,30
2,PDF token 500,2210.1,1713,2502,10


In [27]:
# Generate last-50-char snippets for each PDF strategy
fixed_pdf_endings_md = chunk_endings_markdown(
    "PDF fixed 500", fixed_chunks_500_pdf, n=5, tail_chars=50
)

recursive_pdf_endings_md = chunk_endings_markdown(
    "PDF recursive 1000", recursive_chunks_1000_pdf, n=5, tail_chars=50
)

token_pdf_endings_md = chunk_endings_markdown(
    "PDF token 500", token_chunks_500_pdf, n=5, tail_chars=50
)


In [28]:
# Save these snippets into a PDF boundaries .md file

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)

md_path = reports_dir / "pdf_chunking_boundary_examples.md"

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# PDF Chunking Boundary Examples\n\n")
    f.write(
        "This report shows the last 50 characters of the first few chunks for three "
        "strategies applied to the PDF:\n"
        "- PDF fixed 500\n"
        "- PDF recursive 1000\n"
        "- PDF token 500\n\n"
    )

    f.write(fixed_pdf_endings_md)
    f.write("\n\n")
    f.write(recursive_pdf_endings_md)
    f.write("\n\n")
    f.write(token_pdf_endings_md)
    f.write("\n")

print("Saved PDF chunk boundary examples to:", md_path)


Saved PDF chunk boundary examples to: reports\pdf_chunking_boundary_examples.md


## Step 7: Analyze Chunk Quality

In [29]:
def pct_clean_boundaries(chunks):
    clean = 0
    for c in chunks:
        text = c.strip()
        if not text:
            continue
        if text[-1] in ".?!":
            clean += 1
    return clean / len(chunks) * 100 if chunks else 0.0

In [30]:
print(f"PDF - Fixed 500:     {pct_clean_boundaries(fixed_chunks_500_pdf):.1f}% clean endings")
print(f"PDF - Recursive 1000: {pct_clean_boundaries(recursive_chunks_1000_pdf):.1f}% clean endings")
print(f"PDF - Token 500:      {pct_clean_boundaries(token_chunks_500_pdf):.1f}% clean endings")

PDF - Fixed 500:     0.0% clean endings
PDF - Recursive 1000: 36.7% clean endings
PDF - Token 500:      0.0% clean endings


In [31]:
# Use the canonical splitter names defined earlier
fixed_chunks_500_podcast = fixed_splitter_500.split_text(podcast_text)

recursive_splitter_1000_podcast = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,
)
recursive_chunks_1000_podcast = recursive_splitter_1000_podcast.split_text(podcast_text)

token_chunks_500_podcast = token_splitter_500.split_text(podcast_text)

print(f"Podcast - Fixed 500:     {pct_clean_boundaries(fixed_chunks_500_podcast):.1f}% clean endings")
print(f"Podcast - Recursive 1000: {pct_clean_boundaries(recursive_chunks_1000_podcast):.1f}% clean endings")
print(f"Podcast - Token 500:      {pct_clean_boundaries(token_chunks_500_podcast):.1f}% clean endings")


Podcast - Fixed 500:     6.1% clean endings
Podcast - Recursive 1000: 11.8% clean endings
Podcast - Token 500:      28.6% clean endings
